# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source

The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Access basic metadata (as attributes, not as a dict)
print(f"Dataset name: {dataset.metadata.name}\n")
print(f"Dataset description: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}")
print(f"Published date: {dataset.metadata.datePublished}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

In [ ]:
# List all available record sets and their fields
print("Available record sets:\n")
for recset in dataset.metadata.recordSets:
    print(f"- RecordSet name: {recset.name} | @id: {recset.id}")
    print("  Fields:")
    for fld in recset.fields:
        col_ids = [c.id for c in fld.columns] if getattr(fld, 'columns', None) else []
        print(f"    - {fld.name} | @id: {fld.id} | columns: {col_ids}")
    print()

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. We'll use the record set and field `@id`s listed above.

In [ ]:
# Get the record set IDs for extraction
record_set_ids = [recset.id for recset in dataset.metadata.recordSets]
dataframes = {}
for record_set_id in record_set_ids:
    # Iterate through records and convert to DataFrame
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Display columns and a preview from the first record set
if record_set_ids:
    print(f"Available columns in {record_set_ids[0]}:")
    print(dataframes[record_set_ids[0]].columns.tolist())
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. We'll use field/column `@id`s. Adjust the `numeric_field_id` or `groupby_field_id` as needed based on the output above.

In [ ]:
# EDA: Choose first record set and attempt basic filtering/normalizing on a numeric field

main_record_set = record_set_ids[0]
df = dataframes[main_record_set]
print(f"DataFrame loaded for RecordSet @id={main_record_set}, shape: {df.shape}")

# Display a few columns for reference
print("Columns:", df.columns.tolist())

# Try to automatically select a numeric field to analyze
numeric_field_candidates = df.select_dtypes(include=['float', 'int']).columns.tolist()
if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]   # Use first numeric field
    print(f"Using numeric field: {numeric_field_id}")
    
    # Example filter threshold
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a candidate categorical field
    categorical_fields = df.select_dtypes(include=['object']).columns.tolist()
    if categorical_fields:
        groupby_field_id = categorical_fields[0]
        grouped_df = filtered_df.groupby(groupby_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {groupby_field_id}, showing mean {numeric_field_id} per group:")
        display(grouped_df.head())
else:
    print("No numeric fields detected in this record set.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization of the numeric field, if available
if numeric_field_candidates:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot, optionally by group
    if categorical_fields:
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=groupby_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {groupby_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization skipped: No numeric fields available.")

## 6. Conclusion

In this notebook, we've:
- Loaded the FAIR^2 Croissant-conformant dataset and explored its metadata.
- Identified and loaded record sets and fields using their `@id`s.
- Extracted tabular data for analysis and previewed the main structures.
- Carried out exploratory data analysis with filtering, normalization, and grouping on a numeric field.
- Visualized key distributions and group summaries.

Refer to the [mlcroissant documentation](https://mlcommons.github.io/croissant/python/latest/) and the dataset’s Croissant schema for further field- and ID-specific exploration. Adapt analysis to your research focus by utilizing other record sets or fields as needed.
